In [4]:
import librosa
import soundfile
import os, glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [5]:
def extract_feature(file_name, mfcc, chroma, mel):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate

    if X.ndim > 1:
        X = X.mean(axis=1)

    if sample_rate != 22050:
        X = librosa.resample(X, orig_sr=sample_rate, target_sr=22050)
        sample_rate = 22050

    if chroma:
        stft = np.abs(librosa.stft(X))
    result = np.array([])
    if mfcc:
        mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
        result = np.hstack((result, mfccs))
    if chroma:
        chroma_f = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
        result = np.hstack((result, chroma_f))
    if mel:
        mel_f = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
        result = np.hstack((result, mel_f))
    return result

In [6]:
emotions = {
    '01': 'neutral', 
    '02': 'calm', 
    '03': 'happy', 
    '04': 'sad',
    '05': 'angry', 
    '06': 'fearful', 
    '07': 'disgust', 
    '08': 'surprised'
}
observed_emotions = ['calm', 'happy', 'fearful', 'disgust']


def load_data(test_size=0.2):
    x, y = [], []
    for file in glob.glob("speech-emotion-recognition-ravdess-data/Actor_*/*.wav"):
        file_name = os.path.basename(file)
        emotion = emotions[file_name.split("-")[2]]
        if emotion not in observed_emotions:
            continue
        feature = extract_feature(file, mfcc=True, chroma=True, mel=True)
        x.append(feature)
        y.append(emotion)
    return train_test_split(np.array(x), y, test_size=test_size, random_state=9)

In [7]:
#DataFlair - Split the dataset
x_train,x_test,y_train,y_test=load_data(test_size=0.25)

In [8]:
print((x_train.shape[0], x_test.shape[0]))
print(f"Features extracted: {x_train.shape[1]}")

(576, 192)
Features extracted: 180


In [9]:


model = make_pipeline(
    StandardScaler(),
    MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08,
                  hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)
)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_true=y_test, y_pred=y_pred)
print("Accuracy: {:.2f}%".format(accuracy * 100))

joblib.dump(model, "emotion_model.pkl")

Accuracy: 78.12%


['emotion_model.pkl']

In [10]:
accuracy = accuracy_score(y_true=y_test, y_pred=y_pred)
print("Accuracy: {:.2f}%".format(accuracy * 100))

Accuracy: 78.12%
